In [ ]:
# Core Libraries
!pip -q install streamlit
!pip -q install pyngrok

# LangChain
!pip -q install langchain
!pip -q install langchain-community
!pip -q install langchain-huggingface

# AI
!pip -q install sentence-transformers
!pip -q install faiss-cpu

# PDF
!pip -q install pypdf
!pip -q install python-docx

# Gemini
!pip -q install google-genai

In [ ]:
import os
import streamlit as st
import faiss

from google.colab import files

from langchain_community.document_loaders import (
    PyPDFLoader,
    TextLoader,
    Docx2txtLoader,
)

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_huggingface import HuggingFaceEmbeddings

from langchain_community.vectorstores import FAISS

from google import genai

print("✅ Libraries Imported Successfully")

In [ ]:
# Paste your API key here

GEMINI_API_KEY = "YOUR_GEMINI_API_KEY"

client = genai.Client(api_key=GEMINI_API_KEY)

print("✅ Gemini Connected Successfully")

In [ ]:
response = client.models.generate_content(
    model="gemini-flash-latest",
    contents="Explain Artificial Intelligence in simple words."
)

print(response.text)

In [ ]:
from google import genai

# Create client
client = genai.Client(api_key=GEMINI_API_KEY)

# Generate response
response = client.models.generate_content(
    model="gemini-flash-latest",
    contents="Explain Artificial Intelligence in simple words."
)

print(response.text)

In [ ]:
print("========== PROJECT STATUS ==========")
print("✅ Streamlit Installed")
print("✅ LangChain Installed")
print("✅ FAISS Installed")
print("✅ HuggingFace Ready")
print("✅ Gemini Connected")
print("====================================")

In [ ]:
# ==========================================
# Import Libraries
# ==========================================

from google.colab import files
from langchain_community.document_loaders import PyPDFLoader
import os

print("✅ Libraries Imported")

In [ ]:
# ==========================================
# Upload PDF
# ==========================================

uploaded = files.upload()

pdf_file = list(uploaded.keys())[0]

print("Uploaded File :", pdf_file)

In [ ]:
# ==========================================
# Read PDF
# ==========================================

loader = PyPDFLoader(pdf_file)

documents = loader.load()

print("✅ PDF Loaded Successfully")
print("Total Pages :", len(documents))

In [ ]:
# ==========================================
# First Page
# ==========================================

print(documents[0].page_content)

In [ ]:
# ==========================================
# Combine All Pages
# ==========================================

full_text = ""

for page in documents:
    full_text += page.page_content + "\n"

print("Total Characters :", len(full_text))

In [ ]:
# ==========================================
# PDF Statistics
# ==========================================

pages = len(documents)
words = len(full_text.split())
characters = len(full_text)

print("=" * 40)
print("PDF STATISTICS")
print("=" * 40)

print("Pages      :", pages)
print("Words      :", words)
print("Characters :", characters)

In [ ]:
# ==========================================
# Preview
# ==========================================

print(full_text[:3000])

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

print("✅ Text Splitter Imported Successfully")

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", " ", ""]
)

print("✅ Text Splitter Created")

In [ ]:
chunks = text_splitter.split_text(full_text)

print("✅ Document Split Successfully")
print("Total Chunks:", len(chunks))

In [ ]:
print("=" * 80)
print(chunks[0])
print("=" * 80)

In [ ]:
print("=" * 80)
print(chunks[1])
print("=" * 80)

In [ ]:
print("=" * 50)
print("CHUNK STATISTICS")
print("=" * 50)

print("Total Chunks :", len(chunks))

sizes = [len(chunk) for chunk in chunks]

print("Largest Chunk :", max(sizes))
print("Smallest Chunk:", min(sizes))
print("Average Chunk :", sum(sizes) / len(sizes))

In [ ]:
for i in range(min(5, len(chunks))):
    print("=" * 80)
    print(f"Chunk {i+1}")
    print("=" * 80)
    print(chunks[i][:500])
    print("\n")

In [ ]:
# ==========================================
# Import Embedding Model
# ==========================================

from langchain_huggingface import HuggingFaceEmbeddings

print("✅ Embedding Library Imported")

In [ ]:
# ==========================================
# Load Embedding Model
# ==========================================

embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5"
)

print("✅ Embedding Model Loaded Successfully")

In [ ]:
test_vector = embedding_model.embed_query(
    "What is Machine Learning?"
)

print("Embedding Length :", len(test_vector))

In [ ]:
print("Generating embeddings...")

chunk_embeddings = embedding_model.embed_documents(chunks)

print("✅ Embeddings Generated Successfully")
print("Total Embeddings :", len(chunk_embeddings))

In [ ]:
print(chunk_embeddings[0][:20])

In [ ]:
print("="*50)

print("Total Chunks :", len(chunks))

print("Total Embeddings :", len(chunk_embeddings))

print("Dimension :", len(chunk_embeddings[0]))

print("="*50)

In [ ]:
sentence1 = "Machine Learning is a branch of Artificial Intelligence."

sentence2 = "AI includes Machine Learning."

vec1 = embedding_model.embed_query(sentence1)

vec2 = embedding_model.embed_query(sentence2)

print("Sentence 1 Vector Length:", len(vec1))
print("Sentence 2 Vector Length:", len(vec2))

In [ ]:
# ==========================================
# Import FAISS
# ==========================================

from langchain_community.vectorstores import FAISS

print("✅ FAISS Imported Successfully")

In [ ]:
# ==========================================
# Create FAISS Vector Database
# ==========================================

vector_db = FAISS.from_texts(
    texts=chunks,
    embedding=embedding_model
)

print("✅ Vector Database Created Successfully")

In [ ]:
# ==========================================
# Save Vector Database
# ==========================================

vector_db.save_local("vectorstore")

print("✅ Vector Database Saved")

In [ ]:
# ==========================================
# Load Saved Database
# ==========================================

loaded_db = FAISS.load_local(
    "vectorstore",
    embedding_model,
    allow_dangerous_deserialization=True
)

print("✅ Database Loaded Successfully")

In [ ]:
query = "What is Machine Learning?"

results = loaded_db.similarity_search(
    query,
    k=3
)

print("Top Results Found:", len(results))

In [ ]:
for i, doc in enumerate(results, start=1):
    print("=" * 80)
    print(f"Result {i}")
    print("=" * 80)
    print(doc.page_content[:800])
    print("\n")

In [ ]:
query = "Explain Deep Learning"

results = loaded_db.similarity_search(
    query,
    k=2
)

for i, doc in enumerate(results, start=1):
    print("=" * 60)
    print(f"Match {i}")
    print("=" * 60)
    print(doc.page_content[:600])

In [ ]:
results = loaded_db.similarity_search_with_score(
    "What is Artificial Intelligence?",
    k=3
)

for i, (doc, score) in enumerate(results, start=1):
    print("=" * 70)
    print(f"Result {i}")
    print(f"Distance Score: {score:.4f}")
    print("=" * 70)
    print(doc.page_content[:500])
    print()

In [ ]:
# ==========================================
# Import Libraries
# ==========================================

from google import genai

print("✅ Libraries Imported")

In [ ]:
# ==========================================
# Gemini Client
# ==========================================

GEMINI_API_KEY = "Enter Gemini API Key:"

client = genai.Client(api_key=GEMINI_API_KEY)

print("✅ Gemini Connected")

In [ ]:
# ==========================================
# Create Retriever
# ==========================================

retriever = loaded_db.as_retriever(
    search_kwargs={"k": 3}
)

print("✅ Retriever Ready")

In [ ]:
# ==========================================
# RAG Function
# ==========================================

def ask_ai(question):

    docs = retriever.invoke(question)

    context = "\n\n".join(
        [doc.page_content for doc in docs]
    )

    prompt = f"""
You are an AI Study Assistant.

Answer ONLY from the provided context.

If the answer is not found in the context, reply:

"I couldn't find the answer in the uploaded document."

Context:
{context}

Question:
{question}

Answer:
"""

    response = client.models.generate_content(
        model="gemini-3.1-flash-lite",
        contents=prompt
    )

    return response.text

In [ ]:
question = "What is Machine Learning?"

answer = ask_ai(question)

print(answer)

In [ ]:
while True:

    question = input("\nAsk Question (type exit to quit): ")

    if question.lower() == "exit":
        break

    answer = ask_ai(question)

    print("\nAI Answer:\n")

    print(answer)

In [ ]:
def show_context(question):

    docs = retriever.invoke(question)

    print("="*80)

    for i, doc in enumerate(docs):

        print(f"Chunk {i+1}")

        print("-"*80)

        print(doc.page_content[:700])

        print()

In [ ]:
show_context("Explain Neural Network")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:


import streamlit as st
import tempfile

from google import genai

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# ---------------------------------------
# PAGE CONFIG
# ---------------------------------------

st.set_page_config(
    page_title="AI Study Assistant",
    page_icon="🎓",
    layout="wide"
)

st.title("🎓 AI Powered Study Assistant")

st.write(
    "Upload your PDF and ask questions using Gemini AI."
)

st.divider()

# ---------------------------------------
# GEMINI API
# ---------------------------------------

api_key = st.text_input(
    "Enter Gemini API Key",
    type="password"
)

client = None

if api_key:

    try:

        client = genai.Client(api_key=api_key)

        st.success("✅ Gemini Connected")

    except Exception as e:

        st.error(str(e))

st.divider()

# ---------------------------------------
# PDF Upload
# ---------------------------------------

uploaded_file = st.file_uploader(
    "Upload PDF",
    type=["pdf"]
)

vector_db = None
embedding_model = None

In [ ]:
!ls

In [ ]:
!streamlit run app.py &>/content/logs.txt &

In [ ]:
from pyngrok import ngrok

ngrok.set_auth_token('3H5JLOveOj2xBPTr1iOOYC1ZFUO_9EQV2goin1ugCgjEZgnZ')

url = ngrok.connect(8501)

print(url)

In [ ]:
!pkill streamlit

In [ ]:
!streamlit run app.py &>/content/logs.txt &

In [ ]:
from pyngrok import ngrok

ngrok.kill()

ngrok.set_auth_token('3H5JLOveOj2xBPTr1iOOYC1ZFUO_9EQV2goin1ugCgjEZgnZ')

url = ngrok.connect(8501)

print(url)

In [ ]:
!pkill streamlit

In [ ]:
!streamlit run app.py &>/content/logs.txt &

In [ ]:
from pyngrok import ngrok

ngrok.kill()

ngrok.set_auth_token('3H5JLOveOj2xBPTr1iOOYC1ZFUO_9EQV2goin1ugCgjEZgnZ')

url = ngrok.connect(8501)

print(url)

In [ ]:
!cp app.py app_backup.py

In [ ]:
!python -m py_compile app.py

In [ ]:
%%writefile README.md
# 🎓 AI Powered Study Assistant

## 📌 Overview
AI Powered Study Assistant is a Retrieval-Augmented Generation (RAG) application that allows users to upload PDF documents and ask questions based on the uploaded content. The application retrieves relevant information using FAISS and generates answers using Google Gemini AI.

## 🚀 Features
- Upload PDF documents
- Text extraction
- Text chunking
- HuggingFace Embeddings
- FAISS Vector Database
- Google Gemini AI
- AI Question Answering
- Retrieved Sources
- Chat History
- Streamlit User Interface

## 🛠️ Technologies
- Python
- Streamlit
- Google Gemini API
- LangChain
- HuggingFace Embeddings
- FAISS
- PyPDFLoader

## 📂 Project Structure
AI-Study-Assistant/
├── app.py
├── requirements.txt
└── README.md

## ▶️ Run Locally

pip install -r requirements.txt

streamlit run app.py

## 👨‍💻 Author

Ashish Prajapati

GitHub:
https://github.com/prajapatiashish85

In [ ]:
%%writefile requirements.txt
streamlit
google-genai
langchain
langchain-community
langchain-text-splitters
langchain-huggingface
sentence-transformers
faiss-cpu
pypdf
python-docx

In [ ]:
!git config --global user.name "Ashish Prajapati"
!git config --global user.email "ashishpr8528@gmail.com"

In [ ]:
!git init

In [ ]:
!pwd

In [ ]:
!git add app.py README.md requirements.txt vectorstore logs.txt

In [ ]:
!git commit -m "Initial commit - AI Powered Study Assistant"

In [ ]:
!git add app.py README.md requirements.txt

In [ ]:
!git diff --cached --name-only

In [ ]:
!ls

In [ ]:
!git log --oneline

In [ ]:
!git branch -M main

In [ ]:
!git remote remove origin 2>/dev/null
!git remote add origin https://github.com/prajapatiashish85/AI-Powered-Study-Assistant.git

In [ ]:
!git push -u origin main

In [ ]:
!git checkout --ours README.md
!git add README.md
!git add app.py requirements.txt logs.txt
!git commit -m "Resolve README merge conflict"
!git push -u origin main

In [ ]:
!git add app.py

In [ ]:
!git log --oneline --graph --all -5

In [ ]:
!git push origin main

In [ ]:
!git remote set-url origin https://github.com/prajapatiashish85/-AI-Powered-Study-Assistant.git

In [ ]:
!git remote -v